# Chat buttons and text box

The main notebook runs these cells after creating `advisor`.
Type a message and click **Send**. **New trip** clears the chat.
Turn the image checkbox off if you only want an itinerary.

In [ ]:
"""Notebook UI; executed by the final notebook cell after advisor initialization."""

import ipywidgets as widgets
from IPython.display import HTML, Image, display

chat_output = widgets.Output(layout={"border": "1px solid #dddddd", "padding": "12px"})
message_box = widgets.Textarea(placeholder="Destination, duration and interests, or ask about weather/currency...",
                               description="You:", layout=widgets.Layout(width="100%", height="90px"))
send_button = widgets.Button(description="Send", button_style="primary", icon="paper-plane")
reset_button = widgets.Button(description="New trip", icon="refresh")
planner_selector = widgets.Dropdown(options=list(MODEL_RATES), value=advisor.planner_model, description="Planner:")
image_enabled = widgets.Checkbox(value=True, description="Generate an image for each itinerary")
status = widgets.HTML(value="<i>Ready when you are.</i>")
usage_display = widgets.HTML(value=advisor.usage_summary_html())

### Show markdown


In [ ]:
def show_markdown(text):
    chat_output.append_display_data(HTML(safe_markdown(text)))


### Set busy


In [ ]:
def set_busy(busy):
    for control in (send_button, reset_button, planner_selector, image_enabled):
        control.disabled = busy
    message_box.disabled = busy

### Send message


In [ ]:
def send_message(_):
    user_text = message_box.value.strip()
    if not user_text:
        status.value = "<i>Type a message before sending.</i>"
        return
    set_busy(True)
    status.value = "<i>Planning your response...</i>"
    advisor.planner_model = planner_selector.value
    try:
        show_markdown(f"**You**\n\n{user_text}")
        try:
            reply = advisor.reply(user_text)
        except (RuntimeError, ValueError, KeyError, TypeError):
            show_markdown("**Request failed.** Check API access/quota, shorten your message, or start a new trip. Your input is available to retry.")
            status.value = "<i>Request failed.</i>"
        else:
            show_markdown(f"**Travel Advisor**\n\n{reply}")
            message_box.value = ""
            if advisor.last_itinerary and image_enabled.value:
                status.value = "<i>Generating your destination image...</i>"
                try:
                    image_bytes = advisor.generate_image()
                except (RuntimeError, ValueError):
                    show_markdown("*The itinerary is ready, but image generation failed. Check image-model access and quota. Your itinerary has been saved in the conversation.*")
                else:
                    chat_output.append_display_data(Image(data=image_bytes, format="png", width=512))
                    show_markdown("*AI-generated destination illustration; not a photograph or a map.*")
            status.value = "<i>Ready for your next message.</i>"
    finally:
        usage_display.value = advisor.usage_summary_html()
        set_busy(False)


### Start new trip


In [ ]:
def start_new_trip(_):
    advisor.reset()
    message_box.value = ""
    # Update the widget directly; callback display capture varies by frontend.
    chat_output.outputs = ()
    show_markdown("**Travel Advisor**\n\nHi! Where would you like to go, for how long, and what do you enjoy doing?")
    status.value = "<i>Started a new trip.</i>"
    usage_display.value = advisor.usage_summary_html()


In [ ]:
send_button.on_click(send_message)
reset_button.on_click(start_new_trip)
display(widgets.VBox([chat_output, message_box, widgets.HBox([send_button, reset_button, planner_selector]),
                      image_enabled, usage_display, status]))
start_new_trip(None)